In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "windowing_normalised.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "preprocessing" / "windowing_normalised.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "preprocessing"))

import pandas as pd
import numpy as np
from windowing_normalised import create_windows_normalized

eda_e4   = pd.read_csv('../Data/preprocessed_data/normalized/normalized_EDA_E4.csv')['EDA_E4'].values
bvp_e4   = pd.read_csv('../Data/preprocessed_data/normalized/normalized_BVP_E4.csv')['BVP_E4'].values
hr_e4    = pd.read_csv('../Data/preprocessed_data/normalized/normalized_HR_E4.csv')['HR_E4'].values
temp_e4  = pd.read_csv('../Data/preprocessed_data/normalized/normalized_TEMP_E4.csv')['TEMP_E4'].values

ecg_resp  = pd.read_csv('../Data/preprocessed_data/normalized/normalized_ECG_RB.csv')['ECG_RB'].values
eda_resp  = pd.read_csv('../Data/preprocessed_data/normalized/normalized_EDA_RB.csv')['EDA_RB'].values
emg_resp  = pd.read_csv('../Data/preprocessed_data/normalized/normalized_EMG_RB.csv')['EMG_RB'].values
resp_resp = pd.read_csv('../Data/preprocessed_data/normalized/normalized_RESP_RB.csv')['RESP_RB'].values
temp_resp = pd.read_csv('../Data/preprocessed_data/normalized/normalized_TEMP_RB.csv')['TEMP_RB'].values

print(len(eda_e4), len(bvp_e4), len(hr_e4), len(temp_e4))
print(len(ecg_resp), len(eda_resp), len(emg_resp), len(resp_resp), len(temp_resp))

31494 503943 7715 4442067
4442067 4442067 4442067 4442067 4442067


In [2]:
signals = {
    'EDA_E4': eda_e4, 'BVP_E4': bvp_e4, 'HR_E4': hr_e4, 'TEMP_E4': temp_e4,
    'ECG_RB': ecg_resp, 'EDA_RB': eda_resp, 'EMG_RB': emg_resp,
    'RESP_RB': resp_resp, 'TEMP_RB': temp_resp,
}

fs_dict = {
    'EDA_E4': 4, 'BVP_E4': 64, 'HR_E4': 1, 'TEMP_E4': 4,
    'ECG_RB': 700, 'EDA_RB': 700, 'EMG_RB': 700, 'RESP_RB': 700, 'TEMP_RB': 700,
}

result = create_windows_normalized(signals, fs_dict, window_sec=5, overlap=0.5)

for name, arr in result['windows'].items():
    print(name, arr.shape)

EDA_E4 (3148, 20)
BVP_E4 (3148, 320)
HR_E4 (3856, 5)
TEMP_E4 (444205, 20)
ECG_RB (2537, 3500)
EDA_RB (2537, 3500)
EMG_RB (2537, 3500)
RESP_RB (2537, 3500)
TEMP_RB (2537, 3500)


In [3]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "windowing_normalised.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "preprocessing" / "windowing_normalised.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "preprocessing"))

from windowing_normalised import windows_to_dataframe

for name in signals:
    df = windows_to_dataframe(result, name)
    df.to_csv(f'../Data/preprocessed_data/windows_normalized/windows_normalized_{name}.csv', index=False)

In [4]:
import os, shutil

os.makedirs('../Data/preprocessed_data', exist_ok=True)
os.makedirs('../Data/preprocessed_data/filtered', exist_ok=True)
os.makedirs('../Data/preprocessed_data/normalized', exist_ok=True)
os.makedirs('../Data/preprocessed_data/windows_filtered', exist_ok=True)
os.makedirs('../Data/preprocessed_data/windows_normalized', exist_ok=True)

for f in os.listdir():
    if not f.endswith('.csv'):
        continue
    if f.startswith('windows_normalized_'):
        shutil.move(f, f'../Data/preprocessed_data/windows_normalized/{f}')
    elif f.startswith('windows_'):
        shutil.move(f, f'../Data/preprocessed_data/windows_filtered/{f}')
    elif f.startswith('normalized_'):
        shutil.move(f, f'../Data/preprocessed_data/normalized/{f}')
    elif f.endswith('_filtered.csv') or f.endswith('_filtered_respiban.csv'):
        shutil.move(f, f'../Data/preprocessed_data/filtered/{f}')

print("Done. Contents:")
for root, dirs, files in os.walk('../Data/preprocessed_data'):
    for f in files:
        print(os.path.join(root, f))

Done. Contents:
../Data/preprocessed_data\filtered\bvp_filtered.csv
../Data/preprocessed_data\filtered\ecg_filtered_respiban.csv
../Data/preprocessed_data\filtered\eda_filtered.csv
../Data/preprocessed_data\filtered\eda_filtered_respiban.csv
../Data/preprocessed_data\filtered\emg_centered_respiban.csv
../Data/preprocessed_data\filtered\hr_filtered.csv
../Data/preprocessed_data\filtered\ibi_filtered.csv
../Data/preprocessed_data\filtered\resp_filtered_respiban.csv
../Data/preprocessed_data\filtered\temp_filtered.csv
../Data/preprocessed_data\filtered\temp_filtered_respiban.csv
../Data/preprocessed_data\normalized\normalized_BVP_E4.csv
../Data/preprocessed_data\normalized\normalized_ECG_RB.csv
../Data/preprocessed_data\normalized\normalized_EDA_E4.csv
../Data/preprocessed_data\normalized\normalized_EDA_RB.csv
../Data/preprocessed_data\normalized\normalized_EMG_RB.csv
../Data/preprocessed_data\normalized\normalized_HR_E4.csv
../Data/preprocessed_data\normalized\normalized_IBI.csv
../Data/

In [5]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "windowing_normalised.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "preprocessing" / "windowing_normalised.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "preprocessing"))

from windowing_normalised import plot_all_windowed_signals
import os

figure_dir = os.path.join('..', 'outputs', '05_windowing_normalised', 'figures')
saved_plots = plot_all_windowed_signals(
    result,
    output_dir=figure_dir,
    prefix='normalized',
    max_windows=5,
    show=False,
)

print('Saved normalized window plots:')
for path in saved_plots:
    print(path)


Saved normalized window plots:
..\outputs\05_windowing_normalised\figures\normalized_EDA_E4.png
..\outputs\05_windowing_normalised\figures\normalized_BVP_E4.png
..\outputs\05_windowing_normalised\figures\normalized_HR_E4.png
..\outputs\05_windowing_normalised\figures\normalized_TEMP_E4.png
..\outputs\05_windowing_normalised\figures\normalized_ECG_RB.png
..\outputs\05_windowing_normalised\figures\normalized_EDA_RB.png
..\outputs\05_windowing_normalised\figures\normalized_EMG_RB.png
..\outputs\05_windowing_normalised\figures\normalized_RESP_RB.png
..\outputs\05_windowing_normalised\figures\normalized_TEMP_RB.png
